In [2]:
!pip install cv2

ERROR: Could not find a version that satisfies the requirement cv2 (from versions: none)
ERROR: No matching distribution found for cv2


In [3]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import io
import sys
import os

# Import the color configuration
from color_config import COLOR_THRESHOLDS


ModuleNotFoundError: No module named 'cv2'

In [ ]:
def segment_hydrocarbon_uv(image_rgb, config):
    """
    Function to segment hydrocarbon rocks under UV luminescence.
    Uses Color Thresholding and Area Filtering according to the article:
    "Novel Lithology Identification Method for Drilling Cuttings Under PDC Bit".
    
    Args:
        image_rgb (np.ndarray): The input RGB image.
        config (dict): The color thresholds configuration dictionary.
        
    Returns:
        np.ndarray: The final binary mask combined for all detected oil classes.
        np.ndarray: The segmented RGB image.
    """
    # Step 1: Convert RGB to HSV
    # Disassociating Hue and Value strictly isolates the color from light intensity variations and tray reflections.
    hsv_image = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2HSV)

    # Initialize a globally grouped initial mask to handle multiple colors
    combined_initial_mask = np.zeros(hsv_image.shape[:2], dtype=np.uint8)

    # Step 2: Extract bounds and apply Color Thresholding dynamically for each class
    for oil_class, bounds in config.items():
        lower_bound = bounds["lower"]
        upper_bound = bounds["upper"]
        
        # Segment the specific color class
        class_mask = cv2.inRange(hsv_image, lower_bound, upper_bound)
        
        # Combine the masks
        combined_initial_mask = cv2.bitwise_or(combined_initial_mask, class_mask)

    # Step 3: Area Filtering
    # Find contours on the merged initial mask
    contours, _ = cv2.findContours(combined_initial_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Prepare an empty canvas for the final cleaned mask
    final_mask = np.zeros_like(combined_initial_mask)
    
    # Adhere strictly to the academic constraint: Area (P) >= 100 pixels.
    # Discard non-hydrocarbon microscopic artifacts.
    for contour in contours:
        area = cv2.contourArea(contour)
        if area >= 100:  
            # Retain robust hydrocarbon indications
            cv2.drawContours(final_mask, [contour], -1, 255, thickness=cv2.FILLED)

    # Apply the mask to the original image to visualize the isolated rocks
    segmented_rgb = cv2.bitwise_and(image_rgb, image_rgb, mask=final_mask)

    return final_mask, segmented_rgb

In [ ]:
# Mock file upload equivalent for testing purposes.
# Replace with actual image loading logic (e.g., from an uploaded file or path).
# image_path = "path/to/uv_image.jpg"
# image_rgb = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)

def visualize_segmentation(image_rgb):
    """
    Visualize the segmentation results.
    """
    # Execute the segmentation algorithm
    final_mask, segmented_rock = segment_hydrocarbon_uv(image_rgb, COLOR_THRESHOLDS)
    
    # Display the comparisons using Matplotlib
    fig, ax = plt.subplots(1, 3, figsize=(18, 6))
    
    # Plot 1: Original Image
    ax[0].imshow(image_rgb)
    ax[0].set_title("Original UV Photo")
    ax[0].axis('off')
    
    # Plot 2: Binary Mask Result
    ax[1].imshow(final_mask, cmap='gray')
    ax[1].set_title("Binary Mask (Area >= 100 px)")
    ax[1].axis('off')
    
    # Plot 3: Segmented Rock Result
    ax[2].imshow(segmented_rock)
    ax[2].set_title("Segmented Hydrocarbon Rocks")
    ax[2].axis('off')
    
    plt.tight_layout()
    plt.show()

# Example usage (commented out due to missing image file)
# visualize_segmentation(image_rgb)
